In [ ]:
!apt-get install tesseract-ocr -y
!apt-get install tesseract-ocr-fra -y
!apt-get install poppler-utils -y
!pip install flask pyngrok transformers torch pillow pytesseract pdf2image

import os
import torch
import re
from PIL import Image
from flask import Flask, request, jsonify
from pyngrok import ngrok
from werkzeug.utils import secure_filename
from transformers import pipeline
from pdf2image import convert_from_path

# 1. LOAD THE HYBRID BRAIN
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU' if device == 0 else 'CPU'}")

try:
    doc_qa = pipeline("document-question-answering", model="impira/layoutlm-document-qa", device=device)
except Exception as e:
    print(f"Error loading AI model: {e}")
    doc_qa = None

def clean_val(p):
    if not p: return 0.0
    # Clean TND, spaces, and handle the 3-decimal Tunisian format
    p = str(p).replace('TND', '').replace('€', '').replace(' ', '').replace(',', '.')
    match = re.search(r'(\d+\.\d+)', p)
    return float(match.group(1)) if match else 0.0

class HybridExtractor:
    def process(self, file_path):
        # INCREASE RESOLUTION (dpi=300) to see small text at the bottom
        if file_path.lower().endswith('.pdf'):
            images = convert_from_path(file_path, dpi=300)
            temp_image = file_path + ".jpg"
            images[0].save(temp_image, 'JPEG', quality=95)
            image_to_process = temp_image
            is_temp = True
        else:
            image_to_process = file_path
            is_temp = False

        results = {}
        img_obj = Image.open(image_to_process).convert("RGB")
        
        # A. Use Rules (Regex) - PSM 6 is good for tables
        import pytesseract
        text = pytesseract.image_to_string(img_obj, lang='fra+ara', config='--psm 6')
        print("\n" + "="*50)
        print("🔍 FULL OCR TEXT CAPTURED:")
        print(text)
        print("="*50 + "\n")
        
        patterns = {
            'companyName': r'([A-Za-zÀ-ÖØ-öø-ÿ0-9\s]{3,}(?:SARL|SAS|SA|EURL|S\.A\.R\.L\.|S\.A\.S\.))',
            'invoiceNumber': r'\b(FAC-[A-Z0-9-]+)',
            'totalHT': r'(?:Total H\.T|الإجمالي الصافي)[:\s]*([\d\s,.]+)',
            'tvaAmount': r'(?:TVA 19%|الأداء)[:\s]*([\d\s,.]+)',
            'timbre': r'(?:Timbre|معلوم الطابع)[:\s]*([\d\s,.]+)',
            'totalTTC': r'(?:Total T\.T\.C|المجموع الجملي|TOTAL TTC)[:\s]*([\d\s,.]+(?:\s?TND)?)'
        }
        
        for key, p in patterns.items():
            match = re.search(p, text, re.IGNORECASE | re.MULTILINE)
            if match:
                results[key] = match.group(1).strip()

        # B. AI Fallback (DocVQA)
        if doc_qa:
            ai_questions = {
                'companyName': "What is the name of the company at the very top left?",
                'client': "Who is the client name after 'CLIENT /'?",
                'totalTTC': "What is the final TOTAL T.T.C amount?",
                'totalHT': "What is the Total H.T value?",
                'tvaAmount': "What is the TVA 19% amount?",
                'invoiceNumber': "What is the N° of the invoice?"
            }
            for key, q in ai_questions.items():
                if not results.get(key):
                    print(f"🤖 AI Thinking: {q}")
                    try:
                        ans = doc_qa(img_obj, q)
                        if ans: results[key] = ans[0]['answer']
                    except: pass

        if is_temp: os.remove(temp_image)
        return results

# 2. FLASK API
app = Flask(__name__)
UPLOAD_FOLDER = '/content/api_uploads'
os.makedirs(UPLOAD_FOLDER, exist_ok=True)
extractor = HybridExtractor()

@app.route('/api/extract', methods=['POST'])
def extract_invoice_api():
    file = request.files.get('file')
    if not file: return jsonify({"error": "No file"}), 400
    
    filename = secure_filename(file.filename)
    filepath = os.path.join(UPLOAD_FOLDER, filename)
    file.save(filepath)
    
    try:
        data = extractor.process(filepath)
        
        # Mapping to clean numbers
        response = {
            "companyName": data.get("companyName", "Unknown"),
            "invoiceNumber": data.get("invoiceNumber", "Pending"),
            "date": "Extracted", 
            "totalAmount": clean_val(data.get("totalTTC")),
            "totalHT": clean_val(data.get("totalHT")),
            "tvaAmount": clean_val(data.get("tvaAmount")),
            "timbre": clean_val(data.get("timbre")),
            "client": data.get("client", ""),
            "confidenceScores": { "overall": 1.0 }
        }
        
        os.remove(filepath)
        return jsonify(response), 200
    except Exception as e:
        if os.path.exists(filepath): os.remove(filepath)
        return jsonify({"error": str(e)}), 500

port = 5000
ngrok.set_auth_token("3CwVPLjm9hPafHznWlb0EhcM6bu_6YX8ZbBhwf9kXHQEMX6rC")
public_url = ngrok.connect(port).public_url
print(f"\n🚀 HYBRID FULL-VISION API ONLINE!\n👉 URL: {public_url}/api/extract\n")
app.run(port=port)

